# D3 Flood Analysis Workflow — GeoPandas

Flood exposure screening using GeoPandas spatial operations.


In [1]:
from __future__ import annotations

import json, os

from pathlib import Path

from urllib.error import URLError

from urllib.request import Request, urlopen

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt

import numpy as np



_ROOT = Path.cwd()
if (_ROOT / "examples" / "notebooks").exists():
    OUTPUT_DIR = _ROOT / "examples" / "notebooks" / "geopandas" / "outputs"
else:
    OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALLOW_LIVE_API = os.getenv("GEOPROMPT_ALLOW_LIVE_API", "0") == "1"



def fetch_json(url, fallback):

    if not ALLOW_LIVE_API:

        return fallback

    try:

        req = Request(url, headers={"User-Agent": "geoprompt-notebook/2.0"})

        with urlopen(req, timeout=6) as r:

            return json.loads(r.read().decode("utf-8"))

    except (URLError, TimeoutError, ValueError):

        return fallback



def fetch_first_json(urls, validator, fallback):

    for url in urls:

        payload = fetch_json(url, None)

        if payload is not None and validator(payload):

            return payload, url, True

    return fallback, "fallback", False



import geopandas as gpd

from shapely.geometry import Point, Polygon, box

print(f"geopandas {gpd.__version__} ready")


geopandas 1.1.3 ready


## Section A: Pull Data Sources


In [2]:
flood = {"features": [{"id": "fallback-flood"}]}
weather = {"properties": {"forecast": "fallback"}}
forecast = {"hourly": {"temperature_2m": [0.0]}}

flood, flood_src, flood_live = fetch_first_json(
    [
        "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson",
        "https://api.github.com/repos/OSGeo/gdal",
    ],
    lambda d: isinstance(d, dict) and bool(d.get("features") or d.get("id")),
    flood,
)
weather, wx_src, wx_live = fetch_first_json(
    [
        "https://api.weather.gov/points/29.76,-95.37",
        "https://api.weather.gov/points/30.27,-97.74",
    ],
    lambda d: isinstance(d, dict) and bool(d.get("properties", {}).get("forecast")),
    weather,
)
forecast, fc_src, fc_live = fetch_first_json(
    [
        "https://api.open-meteo.com/v1/forecast?latitude=29.76&longitude=-95.37&hourly=temperature_2m&forecast_days=1",
        "https://api.open-meteo.com/v1/forecast?latitude=30.27&longitude=-97.74&hourly=temperature_2m&forecast_days=1",
    ],
    lambda d: isinstance(d, dict) and len(d.get("hourly", {}).get("temperature_2m", [])) > 0,
    forecast,
)

flood_count = len(flood.get("features", [])) if isinstance(flood, dict) else 0
if flood_count == 0 and isinstance(flood, dict) and flood.get("id"):
    flood_count = 1
print(f"Flood records: {flood_count} | live={flood_live} | source={flood_src}")
print(f"NOAA forecast exists: {bool(weather.get('properties', {}).get('forecast'))} | live={wx_live} | source={wx_src}")
print(f"Open-Meteo hourly points: {len(forecast.get('hourly', {}).get('temperature_2m', []))} | live={fc_live} | source={fc_src}")


Flood records: 1 | live=False | source=fallback
NOAA forecast exists: True | live=False | source=fallback
Open-Meteo hourly points: 1 | live=False | source=fallback


## Section B: Spatial Analysis (GeoPandas)


In [3]:
assets_data = {

    "asset_id":        ["A1",   "A2",   "A3",   "A4",   "A5"],

    "zone":            ["X",    "AE",   "AE",   "VE",   "AE"],

    "replacement_musd":[4.0,    7.5,    3.2,    9.1,    5.5],

    "elevation_m":     [11.0,   7.0,    8.0,    5.9,    6.5],

}

lons = [-95.42, -95.35, -95.29, -95.26, -95.38]

lats = [ 29.81,  29.76,  29.73,  29.68,  29.71]



ZONE_RISK = {"VE": 1.0, "AE": 0.8, "X": 0.2}

gdf = gpd.GeoDataFrame(assets_data, geometry=gpd.points_from_xy(lons, lats), crs="EPSG:4326")

gdf["zone_risk"] = gdf["zone"].map(ZONE_RISK).fillna(0.3)

gdf["flood_risk_score"] = (gdf["zone_risk"] * 0.7 + (10.0 / gdf["elevation_m"]) * 0.3).round(4)

gdf["expected_loss_musd"] = (gdf["replacement_musd"] * gdf["flood_risk_score"] * 0.18).round(4)

print("Asset flood risk:")

print(gdf[["asset_id","zone","flood_risk_score","expected_loss_musd"]].to_string(index=False))



# Flood zone polygons

flood_zones = gpd.GeoDataFrame(

    {"zone_id": ["Z_AE", "Z_VE"],

     "zone_type": ["AE", "VE"]},

    geometry=[box(-95.40, 29.69, -95.25, 29.78), box(-95.30, 29.65, -95.22, 29.73)],

    crs="EPSG:4326",

)



# 1. Spatial join: assets within flood zones

in_zone = gpd.sjoin(gdf, flood_zones, how="left", predicate="within")

print("\nAssets in flood zones:")

print(in_zone[["asset_id","zone_type"]].to_string(index=False))



# 2. Buffer: flood exposure buffers around assets (projected)

gdf_proj = gdf.to_crs("EPSG:3857")

gdf_proj["buffer_geom"] = gdf_proj.buffer(3000)  # 3 km

print(f"\nExposure buffers (3km): {len(gdf_proj)}")



# 3. Nearest join: assign each asset to nearest zone centroid (km)

flood_zone_centroids = flood_zones.copy()

flood_zone_centroids.geometry = flood_zones.centroid

assets_m = gdf.to_crs("EPSG:3857")

zone_centroids_m = flood_zone_centroids.to_crs("EPSG:3857")

nearest_zone_m = gpd.sjoin_nearest(assets_m, zone_centroids_m, how="left", distance_col="dist_m")

nearest_zone = nearest_zone_m.to_crs("EPSG:4326")

nearest_zone["dist_km"] = nearest_zone_m["dist_m"] / 1000.0

print("\nNearest flood zone per asset (km):")

print(nearest_zone[["asset_id","zone_id","dist_km"]].to_string(index=False))



# 4. Clip assets to AE zone extent

ae_mask = flood_zones[flood_zones["zone_type"] == "AE"]

clipped = gpd.clip(gdf, ae_mask)

print(f"\nAssets within AE zone extent: {list(clipped['asset_id'])}")



# 5. Dissolve: total exposure by zone type

in_zone_clean = in_zone.dropna(subset=["zone_type"])

if len(in_zone_clean) > 0:

    dissolved = in_zone_clean.dissolve(by="zone_type", aggfunc={"replacement_musd": "sum", "flood_risk_score": "mean"})

    print("\nExposure dissolved by zone type:")

    print(dissolved[["replacement_musd","flood_risk_score"]].to_string())



# 6. Bounding box filter

high_risk_area = gdf.cx[-95.40:-95.25, 29.68:29.78]

print(f"\nAssets in high-risk area: {list(high_risk_area['asset_id'])}")



# 7. Overlay: find intersection of asset buffers with flood zones

gdf_buf = gpd.GeoDataFrame(gdf, geometry=gdf.to_crs("EPSG:3857").buffer(5000).to_crs("EPSG:4326"))

overlapping = gpd.overlay(gdf_buf[["asset_id","geometry"]], flood_zones[["zone_id","geometry"]], how="intersection")

print(f"\nBuffer-zone intersections: {len(overlapping)} areas")



gdf.to_file(str(OUTPUT_DIR / "d3-gpd-assets.geojson"), driver="GeoJSON")

print("\nWrote d3-gpd-assets.geojson")



# Visualization

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gdf.plot(ax=axes[0], column="flood_risk_score", cmap="Blues", markersize=160, legend=True,

         legend_kwds={"label": "Flood Risk"})

flood_zones.boundary.plot(ax=axes[0], color="red", linewidth=2, linestyle="--", label="Flood Zones")

for _, row in gdf.iterrows():

    axes[0].annotate(row["asset_id"], (row.geometry.x, row.geometry.y),

                     textcoords="offset points", xytext=(4,4), fontsize=9)

axes[0].set_title("Asset Flood Risk (dashed=flood zones)")

axes[0].legend(); axes[0].grid(True, alpha=0.3)



gdf_s = gdf.sort_values("flood_risk_score", ascending=False)

axes[1].barh(gdf_s["asset_id"], gdf_s["flood_risk_score"], color="#1d4ed8")

axes[1].set_xlabel("Flood Risk Score"); axes[1].set_title("Asset Risk Ranking")

axes[1].grid(True, axis="x", alpha=0.3)

plt.suptitle("D3 Flood: GeoPandas Spatial Analysis", fontweight="bold")

plt.tight_layout(); plt.show()

# Basemap snapshot (real tiled basemap, saved as HTML)
try:
    import folium
    label_candidates = ["node", "stand_id", "asset_id", "zone_id"]
    label_col = next((c for c in label_candidates if c in gdf.columns), gdf.columns[0])
    center_lat = float(gdf.geometry.y.mean())
    center_lon = float(gdf.geometry.x.mean())
    fmap = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles="CartoDB positron")

    for _, row in gdf.iterrows():
        folium.CircleMarker(
            location=[float(row.geometry.y), float(row.geometry.x)],
            radius=6,
            color="#1d4ed8",
            fill=True,
            fill_opacity=0.85,
            popup=f"{label_col}: {row[label_col]}",
        ).add_to(fmap)

    if "demand_gdf" in locals():
        for _, row in demand_gdf.iterrows():
            folium.Marker(
                location=[float(row.geometry.y), float(row.geometry.x)],
                icon=folium.Icon(color="blue", icon="bolt", prefix="fa"),
                popup=f"demand_id: {row.get('demand_id', 'n/a')}",
            ).add_to(fmap)

    if "stations_gdf" in locals():
        for _, row in stations_gdf.iterrows():
            folium.Marker(
                location=[float(row.geometry.y), float(row.geometry.x)],
                icon=folium.Icon(color="green", icon="tree", prefix="fa"),
                popup=f"station_id: {row.get('station_id', 'n/a')}",
            ).add_to(fmap)

    if "incidents_gdf" in locals():
        for _, row in incidents_gdf.iterrows():
            folium.Marker(
                location=[float(row.geometry.y), float(row.geometry.x)],
                icon=folium.Icon(color="red", icon="warning-sign", prefix="glyphicon"),
                popup=f"inc_id: {row.get('inc_id', 'n/a')}",
            ).add_to(fmap)

    if "adapt_gdf" in locals():
        for _, row in adapt_gdf.iterrows():
            folium.Marker(
                location=[float(row.geometry.y), float(row.geometry.x)],
                icon=folium.Icon(color="purple", icon="leaf", prefix="fa"),
                popup=f"site_id: {row.get('site_id', 'n/a')}",
            ).add_to(fmap)

    map_path = OUTPUT_DIR / "d3-gpd-basemap.html"
    fmap.save(str(map_path))
    print(f"Wrote basemap snapshot: {map_path.name}")
    fmap
except Exception as exc:
    print(f"Basemap snapshot skipped: {exc}")


Asset flood risk:
asset_id zone  flood_risk_score  expected_loss_musd
      A1    X            0.4127              0.2971
      A2   AE            0.9886              1.3346
      A3   AE            0.9350              0.5386
      A4   VE            1.2085              1.9795
      A5   AE            1.0215              1.0113

Assets in flood zones:
asset_id zone_type
      A1       NaN
      A2        AE
      A3        AE
      A4        VE
      A5        AE

Exposure buffers (3km): 5



Nearest flood zone per asset (km):


asset_id zone_id   dist_km
      A1    Z_AE 14.295282
      A2    Z_AE  4.244945
      A3    Z_AE  3.948556
      A4    Z_VE  1.281359
      A5    Z_AE  6.910522

Assets within AE zone extent: ['A5', 'A3', 'A2']

Exposure dissolved by zone type:
           replacement_musd  flood_risk_score
zone_type                                    
AE                     16.2            0.9817
VE                      9.1            1.2085

Assets in high-risk area: ['A2', 'A3', 'A4', 'A5']

Buffer-zone intersections: 7 areas


C:\Users\Matt\AppData\Local\Temp\ipykernel_13080\3288354145.py:75: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  flood_zone_centroids.geometry = flood_zones.centroid



Wrote d3-gpd-assets.geojson


C:\Users\Matt\AppData\Local\Temp\ipykernel_13080\3288354145.py:173: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Wrote basemap snapshot: d3-gpd-basemap.html


## Section C: Scenario Comparison


In [4]:
baseline   = {"expected_annual_loss": 18.0, "impacted_assets": 4, "response_time_hours": 8.0}
mitigation = {"expected_annual_loss": 10.5, "impacted_assets": 2, "response_time_hours": 6.0}

metrics = list(baseline.keys())
delta_pct = [round((mitigation[m] - baseline[m]) / abs(baseline[m]) * 100, 1) for m in metrics]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = range(len(metrics))
width = 0.38
axes[0].bar([i - width/2 for i in x], [baseline[m] for m in metrics], width=width, label="Baseline", color="#94a3b8")
axes[0].bar([i + width/2 for i in x], [mitigation[m] for m in metrics], width=width, label="Mitigation", color="#2563eb")
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(metrics, rotation=15)
axes[0].set_title("Flood Mitigation Scenarios"); axes[0].legend(); axes[0].grid(True, axis="y", alpha=0.3)

bar_colors = ["#27ae60" if d < 0 else "#c0392b" for d in delta_pct]
axes[1].barh(metrics, delta_pct, color=bar_colors)
axes[1].axvline(0, color="#555", linewidth=1)
axes[1].set_xlabel("% Change"); axes[1].set_title("Delta % (negative=improvement)")
axes[1].grid(True, axis="x", alpha=0.3)
plt.suptitle("D3 Flood (GeoPandas): Scenario Analysis", fontweight="bold")
plt.tight_layout(); plt.show()

(OUTPUT_DIR / "d3-gpd-complex.json").write_text(
    json.dumps({"baseline": baseline, "mitigation": mitigation, "delta_pct": dict(zip(metrics, delta_pct))},
               indent=2, default=str), encoding="utf-8"
)
print("Wrote d3-gpd-complex.json")
print("Delta % per metric:", dict(zip(metrics, delta_pct)))


Wrote d3-gpd-complex.json
Delta % per metric: {'expected_annual_loss': -41.7, 'impacted_assets': -50.0, 'response_time_hours': -25.0}


C:\Users\Matt\AppData\Local\Temp\ipykernel_13080\1854665166.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
